# 02 | Forecast and evaluate demand

Creates a calendar-aware modeling table, conducts 12 non-overlapping rolling-origin seven-day backtests, compares against a previous-week baseline, retrains on all history, and writes the demonstration forecast. Missing non-Mondays remain unknown, while missing Mondays are treated as regular closures. Historical results: 84 evaluation days, model MAE 4.17, baseline MAE 6.08 (31.51% lower). Re-run in your environment to verify.

In [ ]:
from pyspark.sql import functions as F

SOURCE = "workspace.default.restaurant_daily_demand"
TARGET = "workspace.default.restaurant_modeling"

daily = (
    spark.table(SOURCE)
    .select(
        F.to_date("business_day").alias("business_day"),
        F.col("orders").cast("integer").alias("orders")
    )
)

bounds = daily.agg(
    F.min("business_day").alias("start_date"),
    F.max("business_day").alias("end_date")
).first()

calendar = spark.sql(f"""
    SELECT explode(
        sequence(
            DATE('{bounds.start_date}'),
            DATE('{bounds.end_date}'),
            INTERVAL 1 DAY
        )
    ) AS business_day
""")

modeling = (
    calendar
    .join(daily, "business_day", "left")
    .withColumn(
        "day_of_week",
        F.dayofweek("business_day")
    )
    .withColumn(
        "is_closed_monday",
        (F.col("day_of_week") == 2)
        & F.col("orders").isNull()
    )
    .withColumn(
        "missing_open_day",
        (F.col("day_of_week") != 2)
        & F.col("orders").isNull()
    )
    .withColumn(
        "orders",
        F.when(
            F.col("is_closed_monday"), 0
        ).otherwise(F.col("orders"))
    )
)

modeling.write.format("delta").mode("overwrite").saveAsTable(TARGET)

display(modeling.orderBy("business_day"))

In [ ]:
modeling.select(
    F.count("*").alias("calendar_days"),
    F.sum(
        F.col("is_closed_monday").cast("int")
    ).alias("closed_mondays"),
    F.sum(
        F.col("missing_open_day").cast("int")
    ).alias("missing_open_days"),
    F.sum(
        F.col("orders")
    ).alias("total_orders")
).show()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

pdf = (
    spark.table("workspace.default.restaurant_modeling")
    .orderBy("business_day")
    .toPandas()
)

pdf["business_day"] = pd.to_datetime(pdf["business_day"])
pdf["orders"] = pd.to_numeric(pdf["orders"], errors="coerce")
pdf = pdf.sort_values("business_day").reset_index(drop=True)

features = [
    "weekday", "month", "lag_7",
    "rolling_7", "rolling_28"
]

def add_features(df):
    df = df.copy()
    df["weekday"] = df["business_day"].dt.dayofweek
    df["month"] = df["business_day"].dt.month
    df["lag_7"] = df["orders"].shift(7)
    df["rolling_7"] = (
        df["orders"].shift(1)
        .rolling(7, min_periods=4).mean()
    )
    df["rolling_28"] = (
        df["orders"].shift(1)
        .rolling(28, min_periods=14).mean()
    )
    return df

print("Calendar days:", len(pdf))
print("Recorded demand days:", pdf["orders"].notna().sum())

In [ ]:
def forecast_next_7(history):
    history = history.copy().reset_index(drop=True)

    train_df = add_features(history)
    train_df = train_df.dropna(subset=["orders"])

    model = HistGradientBoostingRegressor(
        max_iter=150,
        max_leaf_nodes=15,
        learning_rate=0.05,
        random_state=42
    )
    model.fit(train_df[features], train_df["orders"])

    forecasts = []

    for _ in range(7):
        next_date = history["business_day"].max() + pd.Timedelta(days=1)

        new_row = pd.DataFrame({
            "business_day": [next_date],
            "orders": [np.nan]
        })

        history = pd.concat(
            [history, new_row],
            ignore_index=True
        )

        history_features = add_features(history)
        current = history_features.iloc[[-1]]

        if next_date.dayofweek == 0:
            prediction = 0
        else:
            prediction = max(
                0,
                round(float(model.predict(current[features])[0]))
            )

        history.loc[history.index[-1], "orders"] = prediction

        forecasts.append({
            "business_day": next_date,
            "predicted_orders": prediction
        })

    return pd.DataFrame(forecasts)


results = []

# Twelve non-overlapping weekly test windows
for window in range(12, 0, -1):
    cutoff = len(pdf) - window * 7

    history = pdf.iloc[:cutoff][["business_day", "orders"]].copy()
    actual = pdf.iloc[cutoff:cutoff + 7][
        ["business_day", "orders"]
    ].copy()

    forecast = forecast_next_7(history)

    comparison = forecast.merge(
        actual,
        on="business_day",
        how="inner"
    )

    comparison["window"] = 13 - window

    results.append(comparison)

backtest = pd.concat(results, ignore_index=True)

# Exclude missing open days, but retain known Monday closures
evaluated = backtest.dropna(subset=["orders"]).copy()

mae = mean_absolute_error(
    evaluated["orders"],
    evaluated["predicted_orders"]
)

print("Backtest windows:", 12)
print("Evaluated days:", len(evaluated))
print("Seven-day recursive forecast MAE:", round(mae, 2))

display(evaluated.tail(14))

In [ ]:
baseline = pdf[["business_day", "orders"]].copy()

baseline["baseline_orders"] = baseline["orders"].shift(7)

comparison = backtest.merge(
    baseline[["business_day", "baseline_orders"]],
    on="business_day",
    how="left"
)

valid = comparison.dropna(
    subset=["orders", "baseline_orders"]
).copy()

model_mae = mean_absolute_error(
    valid["orders"],
    valid["predicted_orders"]
)

baseline_mae = mean_absolute_error(
    valid["orders"],
    valid["baseline_orders"]
)

improvement = (
    (baseline_mae - model_mae) / baseline_mae * 100
    if baseline_mae > 0 else float("nan")
)

print("Evaluated days:", len(valid))
print("Model MAE:", round(model_mae, 2))
print("Baseline MAE:", round(baseline_mae, 2))
print("Improvement (%):", round(improvement, 2))

In [ ]:
save_df = backtest.copy()
save_df["business_day"] = save_df["business_day"].dt.date

spark.createDataFrame(save_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.restaurant_backtest")

print("Backtest saved successfully.")

In [ ]:
# Train final model using all available historical data

final_data = add_features(pdf)

final_train = final_data.dropna(
    subset=["orders"]
).copy()

final_model = HistGradientBoostingRegressor(
    max_iter=150,
    max_leaf_nodes=15,
    learning_rate=0.05,
    random_state=42
)

final_model.fit(
    final_train[features],
    final_train["orders"]
)

print("Final model trained successfully!")
print("Training observations:", len(final_train))
print("Last historical date:", pdf["business_day"].max())

In [ ]:
# Generate final seven-day forecast

history = pdf[["business_day", "orders"]].copy()
final_forecasts = []

for day in range(7):
    next_date = (
        history["business_day"].max()
        + pd.Timedelta(days=1)
    )

    new_row = pd.DataFrame({
        "business_day": [next_date],
        "orders": [np.nan]
    })

    history = pd.concat(
        [history, new_row],
        ignore_index=True
    )

    current = add_features(history).iloc[[-1]]

    if next_date.dayofweek == 0:
        prediction = 0
    else:
        prediction = max(
            0,
            round(float(
                final_model.predict(current[features])[0]
            ))
        )

    history.loc[history.index[-1], "orders"] = prediction

    final_forecasts.append({
        "business_day": next_date,
        "predicted_orders": prediction
    })

final_forecast_df = pd.DataFrame(final_forecasts)

print("Final forecast generated successfully!")
display(final_forecast_df)

In [ ]:
# Save updated forecast to Delta Lake

save_forecast = final_forecast_df.copy()
save_forecast["business_day"] = (
    save_forecast["business_day"].dt.date
)

spark.createDataFrame(save_forecast) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.restaurant_forecast"
    )

print("Final forecast saved successfully!")

display(
    spark.table(
        "workspace.default.restaurant_forecast"
    ).orderBy("business_day")
)